# AutoML Mini-System

A hand-rolled **AutoML** loop: instead of hand-picking one model and tuning it by feel, we
define a **search space** of several model families — each with a small hyperparameter grid —
and let cross-validation decide the winner.

The pipeline for every candidate is the same shape:

$$\text{raw features} \;\to\; \text{StandardScaler} \;\to\; \text{classifier}$$

We then:

1. **Search** every family + grid with 5-fold cross-validation and build a **leaderboard**.
2. **Select** the overall best by mean CV score.
3. **Refit** it on the full training set and **evaluate once** on the held-out test set.

Everything runs on the built-in *breast cancer* dataset — no downloads — in a few seconds.

This is the *skeleton* of what tools like Auto-sklearn / TPOT do; the closing section lists
what a production AutoML system adds on top (meta-learning, ensembling, time budgets).

In [ ]:
import numpy as np                 # numerics
import pandas as pd                # the leaderboard lives in a DataFrame
import matplotlib.pyplot as plt    # bar chart of CV scores
import seaborn as sns              # nicer default styling

# --- dataset + split ---
from sklearn.datasets import load_breast_cancer          # small, built-in, offline
from sklearn.model_selection import train_test_split, GridSearchCV

# --- the pipeline + preprocessing every candidate shares ---
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# --- the model families we will put into competition ---
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# --- final held-out-test metrics ---
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# One seed to make the split, the folds, and the randomized models reproducible.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_style("whitegrid")
print("imports ready")

## 1. Data and the train/test split

We load `load_breast_cancer`: 569 samples, 30 numeric features, binary target
(`0 = malignant`, `1 = benign`).

The **test set is locked away immediately** and touched exactly once at the very end. All
model selection and tuning happens with cross-validation *inside the training set* — the test
set is a final, unbiased estimate of generalization, not part of the search.

In [ ]:
# Load the built-in dataset (ships with scikit-learn, no network access needed).
data = load_breast_cancer()
X, y = data.data, data.target          # X: (569, 30) features, y: (569,) 0/1 labels

# Hold out 25% as the FINAL test set. stratify=y keeps the class balance identical
# in both splits so accuracy / roc_auc are comparable across them.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print(f"features           : {X.shape[1]}")
print(f"train / test sizes : {X_train.shape[0]} / {X_test.shape[0]}")
# class balance = mean of the 0/1 target (fraction of the positive/benign class)
print(f"train class balance: {y_train.mean():.3f}   test class balance: {y_test.mean():.3f}")

## 2. The search space

The heart of an AutoML system is a **declarative search space**: a dictionary mapping each
model family to *(an unfitted estimator, a small grid of hyperparameters to try)*.

Two design points to notice:

- **Every candidate is wrapped in the same `Pipeline`**: `StandardScaler → classifier`. Scaling
  is critical for distance/margin models (SVC, KNN, logistic regression) and harmless for the
  trees. Putting the scaler *inside* the pipeline means it is re-fit on the training folds only
  during cross-validation — no leakage from validation folds into the scaler statistics.
- **Grid keys are prefixed `clf__`**. In a `Pipeline`, a hyperparameter of the step named
  `"clf"` is addressed as `clf__<param>`. So `clf__C` tunes the classifier's `C`.

Grids are deliberately **tiny** so the whole search finishes in seconds — the point is the
mechanism, not an exhaustive sweep.

In [ ]:
# search_space: name -> {"estimator": <unfitted model>, "param_grid": {clf__param: [values]}}
# Each grid is small on purpose (keeps total fits, and runtime, low).
search_space = {
    "LogisticRegression": {
        # linear, probabilistic baseline; C is inverse regularization strength.
        "estimator": LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
        "param_grid": {"clf__C": [0.1, 1.0, 10.0]},                       # 3 configs
    },
    "RandomForest": {
        # bagged decision trees; robust, needs little tuning.
        "estimator": RandomForestClassifier(random_state=RANDOM_STATE),
        "param_grid": {"clf__n_estimators": [100, 200],                   # 2 x 2
                        "clf__max_depth": [None, 5]},                     # = 4 configs
    },
    "GradientBoosting": {
        # sequential boosting of shallow trees; strong on tabular data.
        "estimator": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "param_grid": {"clf__learning_rate": [0.05, 0.1],                 # 2 x 2
                        "clf__max_depth": [2, 3]},                        # = 4 configs
    },
    "SVC": {
        # RBF-kernel support vector machine; VERY sensitive to feature scale (hence the scaler).
        # probability=False, but SVC exposes decision_function, which roc_auc scoring can use.
        "estimator": SVC(kernel="rbf", random_state=RANDOM_STATE),
        "param_grid": {"clf__C": [1.0, 10.0],                            # 2 x 1
                        "clf__gamma": ["scale"]},                         # = 2 configs
    },
    "KNN": {
        # k-nearest-neighbours; distance-based, so scaling matters here too.
        "estimator": KNeighborsClassifier(),
        "param_grid": {"clf__n_neighbors": [3, 5, 7]},                    # 3 configs
    },
}

# Sanity print: how many hyperparameter combinations each family contributes.
for name, cfg in search_space.items():
    n_combos = int(np.prod([len(v) for v in cfg["param_grid"].values()]))
    print(f"{name:20s}: {n_combos} config(s)  x 5 folds = {n_combos * 5} fits")

## 3. Why cross-validation, not a single validation split?

If we carved off *one* validation slice to score candidates, the ranking would ride on the
luck of that particular slice — a small, high-variance sample. **k-fold cross-validation**
rotates the validation role across all $k$ folds and averages, giving a far more stable estimate:

$$\text{CV score} = \frac{1}{k}\sum_{i=1}^{k} \text{score}\big(\text{model trained on folds}\neq i,\ \text{evaluated on fold } i\big)$$

We also keep the **standard deviation** across folds — it tells us how *reliable* each score is.
A model that wins by a hair but with a huge std is not clearly better than the runner-up.

We score with **ROC-AUC** (`scoring="roc_auc"`): threshold-independent and well-behaved on the
mild class imbalance here. `GridSearchCV(..., refit=True)` automatically refits the best config
of each family on the *entire* training set once its search finishes.

In [ ]:
# Driver loop: run one GridSearchCV per model family, collect results into a leaderboard.
results = []              # one row per family for the leaderboard DataFrame
fitted_searches = {}      # keep each fitted GridSearchCV so we can grab the winner later

for name, cfg in search_space.items():
    # Same preprocessing for everyone: scale features, then the family's classifier.
    pipe = Pipeline([
        ("scaler", StandardScaler()),   # step name 'scaler'
        ("clf", cfg["estimator"]),      # step name 'clf'  -> grid keys use the clf__ prefix
    ])

    # 5-fold CV over this family's small grid. refit=True -> best config re-trained on
    # the full training set. n_jobs=-1 parallelises across CPU cores.
    search = GridSearchCV(
        estimator=pipe,
        param_grid=cfg["param_grid"],
        cv=5,
        scoring="roc_auc",
        n_jobs=-1,
        refit=True,
    )
    search.fit(X_train, y_train)        # <-- all the cross-validation work happens here
    fitted_searches[name] = search

    # cv_results_ holds per-config stats; best_index_ points at the winning config's row,
    # so std_test_score[best_index_] is the fold-to-fold std of the BEST config (not the grid).
    best_std = search.cv_results_["std_test_score"][search.best_index_]

    results.append({
        "model": name,
        "best_cv_score": search.best_score_,   # mean CV ROC-AUC of the best config
        "cv_std": best_std,                    # std of that score across the 5 folds
        "best_params": search.best_params_,    # the winning hyperparameters
    })
    print(f"{name:20s} done  CV roc_auc = {search.best_score_:.4f} (+/- {best_std:.4f})")

print("\nall families searched")

## 4. The leaderboard

We assemble the collected results into a DataFrame and sort by mean CV score. This is the
AutoML "report card": every family, its best hyperparameters, and how confidently it scored.

In [ ]:
# Build the leaderboard and sort best-first. reset_index gives a clean 0..n rank order.
leaderboard = (
    pd.DataFrame(results)
    .sort_values("best_cv_score", ascending=False)
    .reset_index(drop=True)
)

# Round the numeric columns for a tidy display (keep the raw values in memory unchanged).
display_board = leaderboard.copy()
display_board["best_cv_score"] = display_board["best_cv_score"].round(4)
display_board["cv_std"] = display_board["cv_std"].round(4)
print("=== LEADERBOARD (sorted by mean CV roc_auc) ===")
print(display_board.to_string(index=True))

In [ ]:
# Bar chart of mean CV score per model, with error bars = +/- 1 std across folds.
fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(
    leaderboard["model"],
    leaderboard["best_cv_score"],
    yerr=leaderboard["cv_std"],      # error bars communicate fold-to-fold reliability
    capsize=6,                        # little caps on the whiskers
    color=sns.color_palette("viridis", len(leaderboard)),
    edgecolor="black",
)

# Zoom the y-axis to where the scores actually live so differences are visible.
lo = (leaderboard["best_cv_score"] - leaderboard["cv_std"]).min()
ax.set_ylim(max(0.0, lo - 0.02), 1.005)
ax.set_ylabel("mean CV ROC-AUC")
ax.set_title("AutoML leaderboard: cross-validated score per model (\u00b11 std)")
# Fix tick positions FIRST, then label them, to avoid a matplotlib FixedLocator warning.
ax.set_xticks(range(len(leaderboard)))
ax.set_xticklabels(leaderboard["model"], rotation=20, ha="right")

# Annotate each bar with its numeric score, placed just above the error bar.
for i, (score, std) in enumerate(zip(leaderboard["best_cv_score"], leaderboard["cv_std"])):
    ax.text(i, score + std + 0.003, f"{score:.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

## 5. Select the winner and evaluate ONCE on the test set

The top row of the leaderboard is our chosen model. Because `GridSearchCV` used
`refit=True`, its `best_estimator_` is *already* a full pipeline trained on the entire training
set with the winning hyperparameters — ready to predict.

We now unlock the test set for the first and only time and report:

- **accuracy** — fraction of correct labels,
- **ROC-AUC** — ranking quality across all thresholds (using `predict_proba` when available,
  otherwise `decision_function`, since our SVC has no probabilities),
- a full **classification report** (precision / recall / F1 per class).

In [ ]:
# Winner = first row of the sorted leaderboard.
best_name = leaderboard.loc[0, "model"]
best_search = fitted_searches[best_name]
best_model = best_search.best_estimator_        # already refit on the FULL training set

print(f"Selected model : {best_name}")
print(f"Best params    : {best_search.best_params_}")
print(f"CV roc_auc     : {best_search.best_score_:.4f}\n")

# Hard label predictions for accuracy and the classification report.
y_pred = best_model.predict(X_test)

# Continuous scores for ROC-AUC: prefer probabilities; fall back to the SVM's decision_function.
if hasattr(best_model, "predict_proba"):
    y_score = best_model.predict_proba(X_test)[:, 1]     # P(class = 1)
else:
    y_score = best_model.decision_function(X_test)       # signed distance to the margin

# --- final held-out-test metrics (the honest generalization estimate) ---
print(f"TEST accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"TEST roc_auc   : {roc_auc_score(y_test, y_score):.4f}\n")
print("classification report:")
print(classification_report(y_test, y_pred, target_names=data.target_names))

## 6. The catch, and what real AutoML adds

**We tuned model selection against the CV score — so the CV winner's score is optimistic.**
By trying many families and grids and then *picking the maximum*, we implicitly fit the
selection procedure to the training folds. The more candidates we try, the more some model wins
partly by luck. This is why the **untouched test set matters**: it is the only estimate that the
search never got to peek at. (Rigorous setups go further with *nested* cross-validation — an
inner CV to tune, an outer CV to score the whole tuning procedure.)

A production **AutoML** system is this loop plus a lot more engineering:

- **Smarter search** — Bayesian optimization / Hyperband instead of brute-force grids, so the
  budget concentrates on promising regions of the space.
- **Meta-learning** — warm-starting the search from configurations that worked well on similar
  past datasets, rather than from scratch.
- **Ensembling** — the final model is usually a weighted blend of the top candidates, not a
  single winner; ensembles typically beat any one model.
- **Time / resource budgets** — an explicit wall-clock or compute limit, with the search
  gracefully returning the best model found so far when the budget runs out.
- **Automated preprocessing** — imputation, encoding, and feature selection are searched over
  too, not fixed to a single `StandardScaler`.

Our mini-system captures the essential skeleton: **a declarative search space, cross-validated
comparison, a leaderboard, and one honest final test.**